In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset , DataLoader
from math import cos , sin
import pandas as pd

In [3]:
class multihead(nn.Module):
  def __init__(self,embed_dim):
    super().__init__()
    self.embed_dim = embed_dim

    self.num_heads = 4
    self.mat_size = self.embed_dim//self.num_heads


    self.querym1 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym1)
    self.keym1 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym1)
    self.valuem1 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem1)


    self.querym2 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym2)
    self.keym2 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym2)
    self.valuem2 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem2)

    self.querym3 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym3)
    self.keym3 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym3)
    self.valuem3 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem3)

    self.querym4 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym4)
    self.keym4 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym4)
    self.valuem4 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem4)

    self.final_weight = nn.Parameter(torch.empty(self.embed_dim,self.embed_dim))
    nn.init.xavier_uniform_(self.final_weight)


  def forward(self,embedding):
    queryA = torch.matmul(embedding, self.querym1)
    keyA =  torch.matmul(embedding, self.keym1)
    valueA = torch.matmul(embedding, self.valuem1)

    queryB = torch.matmul(embedding, self.querym2)
    keyB =  torch.matmul(embedding, self.keym2)
    valueB = torch.matmul(embedding, self.valuem2)

    queryC = torch.matmul(embedding, self.querym3)
    keyC =  torch.matmul(embedding, self.keym3)
    valueC = torch.matmul(embedding, self.valuem3)

    queryD = torch.matmul(embedding, self.querym4)
    keyD =  torch.matmul(embedding, self.keym4)
    valueD = torch.matmul(embedding, self.valuem4)

    #final outputs of each attention

    weightA = torch.matmul(queryA,keyA.transpose(-2,-1))
    weightA = weightA/(self.mat_size**0.5)
    weightA = torch.softmax(weightA,dim = -1)
    embedA = torch.matmul(weightA,valueA)


    weightB = torch.matmul(queryB,keyB.transpose(-2,-1))
    weightB = weightB/(self.mat_size**0.5)
    weightB = torch.softmax(weightB,dim = -1)
    embedB = torch.matmul(weightB,valueB)

    weightC = torch.matmul(queryC,keyC.transpose(-2,-1))
    weightC = weightC/(self.mat_size**0.5)
    weightC = torch.softmax(weightC,dim = -1)
    embedC = torch.matmul(weightC,valueC)

    weightD = torch.matmul(queryD,keyD.transpose(-2,-1))
    weightD = weightD/(self.mat_size**0.5)
    weightD = torch.softmax(weightD,dim = -1)
    embedD = torch.matmul(weightD,valueD)


    final_embed = torch.cat([embedA,embedB,embedC,embedD],dim = -1)
    final_embed = torch.matmul(final_embed,self.final_weight)

    return final_embed


class masked_attention(nn.Module):
  def __init__(self,embed_dim):
    super().__init__()
    self.embed_dim = embed_dim

    self.num_heads = 4
    self.mat_size = self.embed_dim//self.num_heads


    self.querym1 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym1)
    self.keym1 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym1)
    self.valuem1 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem1)


    self.querym2 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym2)
    self.keym2 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym2)
    self.valuem2 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem2)

    self.querym3 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym3)
    self.keym3 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym3)
    self.valuem3 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem3)

    self.querym4 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym4)
    self.keym4 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym4)
    self.valuem4 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem4)

    self.final_weight = nn.Parameter(torch.empty(self.embed_dim,self.embed_dim))
    nn.init.xavier_uniform_(self.final_weight)


  def forward(self,embedding):

    #MULTI ATTENTION
    queryA = torch.matmul(embedding, self.querym1)
    keyA =  torch.matmul(embedding, self.keym1)
    valueA = torch.matmul(embedding, self.valuem1)

    queryB = torch.matmul(embedding, self.querym2)
    keyB =  torch.matmul(embedding, self.keym2)
    valueB = torch.matmul(embedding, self.valuem2)

    queryC = torch.matmul(embedding, self.querym3)
    keyC =  torch.matmul(embedding, self.keym3)
    valueC = torch.matmul(embedding, self.valuem3)

    queryD = torch.matmul(embedding, self.querym4)
    keyD =  torch.matmul(embedding, self.keym4)
    valueD = torch.matmul(embedding, self.valuem4)


    weightA = torch.matmul(queryA,keyA.transpose(-2,-1))
    weightA = weightA/(self.mat_size**0.5)
    maskA = torch.triu(torch.ones(weightA.size(-1),weightA.size(-1),device = weightA.device),1).bool()
    weightA = weightA.masked_fill(maskA,float("-inf"))
    weightA = torch.softmax(weightA,dim = -1)
    embedA = torch.matmul(weightA,valueA)


    weightB = torch.matmul(queryB,keyB.transpose(-2,-1))
    weightB = weightB/(self.mat_size**0.5)
    maskB = torch.triu(torch.ones(weightB.size(-1),weightB.size(-1),device = weightB.device),1).bool()
    weightB = weightB.masked_fill(maskB,float("-inf"))
    weightB = torch.softmax(weightB,dim = -1)
    embedB = torch.matmul(weightB,valueB)

    weightC = torch.matmul(queryC,keyC.transpose(-2,-1))
    weightC = weightC/(self.mat_size**0.5)
    maskC = torch.triu(torch.ones(weightC.size(-1),weightC.size(-1),device = weightC.device),1).bool()
    weightC = weightC.masked_fill(maskC,float("-inf"))
    weightC = torch.softmax(weightC,dim = -1)
    embedC = torch.matmul(weightC,valueC)

    weightD = torch.matmul(queryD,keyD.transpose(-2,-1))
    weightD = weightD/(self.mat_size**0.5)
    maskD = torch.triu(torch.ones(weightD.size(-1),weightD.size(-1),device = weightD.device),1).bool()
    weightD = weightD.masked_fill(maskD,float("-inf"))
    weightD = torch.softmax(weightD,dim = -1)
    embedD = torch.matmul(weightD,valueD)

    final_embed = torch.cat([embedA,embedB,embedC,embedD],dim = -1)

    final_embed = torch.matmul(final_embed,self.final_weight)

    return final_embed

class cross_attention(nn.Module):
  def __init__(self,embed_dim):
    super().__init__()
    self.embed_dim = embed_dim

    self.num_heads = 4
    self.mat_size = self.embed_dim//self.num_heads


    self.querym1 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym1)
    self.keym1 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym1)
    self.valuem1 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem1)


    self.querym2 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym2)
    self.keym2 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym2)
    self.valuem2 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem2)

    self.querym3 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym3)
    self.keym3 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym3)
    self.valuem3 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem3)

    self.querym4 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.querym4)
    self.keym4 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.keym4)
    self.valuem4 = nn.Parameter(torch.empty(self.embed_dim,self.mat_size,requires_grad=True))
    nn.init.xavier_uniform_(self.valuem4)

    self.final_weight = nn.Parameter(torch.empty(self.embed_dim,self.embed_dim))
    nn.init.xavier_uniform_(self.final_weight)

  def forward(self,input,output):
    #MULTI ATTENTION
    queryA = torch.matmul(output, self.querym1) # 1,1,64 64,16    1,16,1
    keyA =  torch.matmul(input, self.keym1) # 1,50,64 , 64,16 50,16
    valueA = torch.matmul(input, self.valuem1)

    queryB = torch.matmul(output, self.querym2)
    keyB =  torch.matmul(input, self.keym2)
    valueB = torch.matmul(input, self.valuem2)

    queryC = torch.matmul(output, self.querym3)
    keyC =  torch.matmul(input, self.keym3)
    valueC = torch.matmul(input, self.valuem3)

    queryD = torch.matmul(output, self.querym4)
    keyD =  torch.matmul(input, self.keym4)
    valueD = torch.matmul(input, self.valuem4)


    weightA = torch.matmul(queryA,keyA.transpose(-2,-1))
    weightA = weightA/(self.mat_size**0.5)
    weightA = torch.softmax(weightA,dim = -1)
    embedA = torch.matmul(weightA,valueA)


    weightB = torch.matmul(queryB,keyB.transpose(-2,-1))
    weightB = weightB/(self.mat_size**0.5)
    weightB = torch.softmax(weightB,dim = -1)
    embedB = torch.matmul(weightB,valueB)

    weightC = torch.matmul(queryC,keyC.transpose(-2,-1))
    weightC = weightC/(self.mat_size**0.5)
    weightC = torch.softmax(weightC,dim = -1)
    embedC = torch.matmul(weightC,valueC)

    weightD = torch.matmul(queryD,keyD.transpose(-2,-1))
    weightD = weightD/(self.mat_size**0.5)
    weightD = torch.softmax(weightD,dim = -1)
    embedD = torch.matmul(weightD,valueD)


    final_embed = torch.cat([embedA,embedB,embedC,embedD],dim = -1)

    final_embed = torch.matmul(final_embed,self.final_weight)

    return final_embed

class Encoder(nn.Module):
  def __init__(self,embed_dim):
    super().__init__()
    self.multihead = multihead(embed_dim)

    self.linear = nn.Sequential(
        nn.Linear(embed_dim,embed_dim*4),
        nn.ReLU(),
        nn.Linear(embed_dim*4,embed_dim)
    )

    self.layernorm1 = nn.LayerNorm(embed_dim)
    self.layernorm2 = nn.LayerNorm(embed_dim)


  def forward(self,embedding):

    final_embed = self.multihead(embedding)

    add = embedding + final_embed

    normmat = self.layernorm1(add)

    linear = self.linear(normmat)

    final_embed = normmat + linear

    norm_final = self.layernorm2(final_embed)

    return norm_final

class Decoder(nn.Module):
  def __init__(self,embed_dim):
    super().__init__()
    self.masked_multihead = masked_attention(embed_dim)
    self.cross_attention = cross_attention(embed_dim)
    
    self.layernorm1 = nn.LayerNorm(embed_dim)
    self.layernorm2 = nn.LayerNorm(embed_dim)
    self.layernorm3 = nn.LayerNorm(embed_dim)

    self.linear = nn.Sequential(
        nn.Linear(embed_dim,embed_dim*4),
        nn.ReLU(),
        nn.Linear(embed_dim*4,embed_dim)
    )



  def forward(self,input,output):

    masking = self.masked_multihead(output)

    add1 = masking + output

    layernorm1 = self.layernorm1(add1)

    cross_attn = self.cross_attention(input,layernorm1)

    add2 = cross_attn + layernorm1

    layernorm2 = self.layernorm2(add2)

    linear = self.linear(layernorm2)

    add3 = layernorm2 + linear

    layernorm3 = self.layernorm3(add3)

    return layernorm3

In [4]:
class Tmr(nn.Module):
  def __init__(self,embed_dim,vocab):
    super().__init__()
    self.embed_dim = embed_dim

    self.encoder1 = Encoder(embed_dim)
    self.encoder2 = Encoder(embed_dim)
    self.encoder3 = Encoder(embed_dim)
    self.encoder4 = Encoder(embed_dim)


    self.decoder1 = Decoder(embed_dim)
    self.decoder2 = Decoder(embed_dim)
    self.decoder3 = Decoder(embed_dim)
    self.decoder4 = Decoder(embed_dim)


    self.e = vocab[0]
    self.m = vocab[1]

    self.eng = nn.Embedding(self.e,self.embed_dim)
    self.mar = nn.Embedding(self.m,self.embed_dim)


    self.final_prob = nn.Sequential(
        nn.Linear(self.embed_dim,self.m)
        #not using softmax due to use of CEloss
    )

  def positional_encodding(self,embedding):
    positional_encodding = []

    elements = int(embedding.size(-2))
    items = int(embedding.size(-1)/2)

    for element in range(elements):
      for i in range(items):

        sine = sin((element)/(10000**(2*i/self.embed_dim)))
        positional_encodding.append(sine)
        cosine = cos((element)/(10000**(2*i/self.embed_dim)))
        positional_encodding.append(cosine)


    positional_encodding = torch.tensor(positional_encodding,dtype = embedding.dtype,device = embedding.device).reshape(-1,element+1,items*2)

    return positional_encodding


  def embedding(self,x,y):

    english = self.eng(x)
    marathi = self.mar(y)

    return english,marathi

  def forward(self,input,output):

    input += self.positional_encodding(input)
    output += self.positional_encodding(output)

    e = self.encoder1(input)
    e = self.encoder2(e)
    e = self.encoder3(e)
    e = self.encoder4(e)


    d = self.decoder1(e,output)
    d = self.decoder2(e,d)
    d = self.decoder3(e,d)
    d = self.decoder4(e,d)


    final_output = self.final_prob(d)

    return final_output



In [5]:
class tokenize:
  def __init__(self,file,limit):
    if limit == "max":
      limit = len(file)
    self.limit = limit
    self.english = file['English']
    self.marathi = file['Marathi']
    self.mar_dict = {"<sos>":0,"<eos>":1,"<null>":2}
    self.eng_dict = {"<sos>":0,"<eos>":1,"<null>":2}
    print("limit = ",limit)

  def processing(self):
    for i,j in zip(self.english,self.marathi):
      i,j = i.replace("!","") , j.replace("!","")
      i,j = i.replace("?","") , j.replace("?","")
      i,j = i.replace(".","") , j.replace(".","")
      i,j = i.replace("'","") , j.replace("'","")
      i = i.lower()

      ekeys = i.split()
      mkeys = j.split()

      for i in ekeys:
        if i not in self.eng_dict:
          self.eng_dict[i] = len(self.eng_dict)

      for j in mkeys:
        if j not in self.mar_dict:
          self.mar_dict[j] = len(self.mar_dict)

    return self.eng_dict , self.mar_dict

  def get_dict(self):
    return self.eng_dict,self.mar_dict

  def dict_size(self):
    return len(self.eng_dict) , len(self.mar_dict)

  def inf_w2n(self,i):
    i = i.replace("!","").replace("?","").replace(".","").replace("'","")
    i = i.lower()

    new = i.split()

    start = [self.eng_dict["<sos>"]]

    value = [self.eng_dict[e] for e in new if e in self.eng_dict]

    end = [self.eng_dict["<eos>"]]

    sent = start + value + end
    while(len(sent)<50):
        sent.append(self.eng_dict["<null>"])

    return sent


  def word_to_num(self):

    eng_item = []
    mar_item = []

    for i,j in zip(self.english,self.marathi):
      if len(eng_item) > self.limit:
        break
      i,j = i.replace("!","") , j.replace("!","")
      i,j = i.replace("?","") , j.replace("?","")
      i,j = i.replace(".","") , j.replace(".","")
      i,j = i.replace("'","") , j.replace("'","")
      i = i.lower()
      eng = i.split()
      mar = j.split()

      #main conversion
      eng_num = [self.eng_dict[val] for val in eng if val in self.eng_dict]
      mar_num = [self.mar_dict[val] for val in mar if val in self.mar_dict]

      evalue = [self.eng_dict["<sos>"]]
      evalue += (eng_num)
      evalue.append(self.eng_dict["<eos>"])

      mvalue = [self.mar_dict["<sos>"]]
      mvalue += (mar_num)
      mvalue.append(self.mar_dict["<eos>"])

      while(len(evalue)<50):
        evalue.append(self.eng_dict["<null>"])
      while(len(mvalue)<50):
        mvalue.append(self.mar_dict["<null>"])

      eng_item.append(evalue)
      mar_item.append(mvalue)

    final_eng = torch.tensor(eng_item)
    final_mar = torch.tensor(mar_item)


    return final_eng,final_mar



In [6]:
from torch.utils.data import DataLoader,Dataset

class MyDataset(Dataset):
  def __init__(self,eng,mar):
    self.eng = eng
    self.mar = mar
  def __len__(self):
    return len(self.mar)
  def __getitem__(self,index):

    return self.eng[index],self.mar[index]

In [8]:
data = pd.read_csv(r"dataset\Transformer-data.csv")


#working with data
data['English'][len(data)-1]

data.tail(10000)
data

,Unnamed: 0,English,Marathi
0,0,Go.,जा.
1,1,Go.,जा.
2,2,Go.,जा.
3,3,Go.,जा.
4,4,Hi.,हाय.
...,...,...,...
118959,118959,There are four main causes of alcohol-related ...,अल्कोहोल-संबंधित मृत्यूची चार मुख्य कारणे आहेत...
118960,118960,There are mothers and fathers who will lie awa...,असे आई आणि वडील आहेत जे मुले झोपी गेल्यानंतर ज...
118961,118961,A carbon footprint is the amount of carbon dio...,कार्बन फूटप्रिंट म्हणजे कार्बन डायऑक्साइड प्रद...
118962,118962,Since there are usually multiple websites on a...,कोणत्याही दिलेल्या विषयावर सहसा अनेक वेबसाइट्स...


In [9]:
token = tokenize(data.tail(30000),"max")

a,b = token.processing()


limit =  30000


In [10]:
lst = []
for i in data["English"]:
  i = i.split()

  lst.append(len(i))
print(max(lst))


token.dict_size()

a = token.inf_w2n("She said that she would follow him no matter where he went.")
print(a)

47
[0, 597, 591, 157, 597, 874, 1448, 542, 435, 1156, 259, 158, 80, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [11]:
from sklearn.model_selection import train_test_split

final_eng,final_mar = token.word_to_num()


X_train , X_test , y_train , y_test = train_test_split(final_eng,final_mar,train_size = 0.80)

train = MyDataset(X_train,y_train)
test = MyDataset(X_test,y_test)

train_loader = DataLoader(train,batch_size=32,shuffle = True)
test_loader = DataLoader(test,batch_size=32,shuffle = True)

In [13]:
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

vocab = token.dict_size()
model = Tmr(64,vocab).to(device)

model.train()
epoch = 30
lr = 0.0001

criterion = nn.CrossEntropyLoss(ignore_index=2)

optimizer = optim.AdamW(model.parameters(),lr = lr)

for e in range(epoch):

  total_loss = 0
  num_batches = 0

  print("epoch --> ",e+1)
  for eng,mar in train_loader:

    eng = eng.to(device)
    mar = mar.to(device)

    eng,mar_input = model.embedding(eng[:,:-1],mar[:,:-1])

    output = model(eng,mar_input)
    optimizer.zero_grad()

    loss = criterion(output.reshape(-1,vocab[1]),mar[:,1:].reshape(-1))

    loss.backward()
    optimizer.step()

    total_loss +=loss.item()
    num_batches +=1


  print(f"AVG_Loss = {total_loss/num_batches}")


cuda
epoch -->  1
AVG_Loss = 7.992313459396362
epoch -->  2
AVG_Loss = 6.7620759906768795
epoch -->  3
AVG_Loss = 6.465435804367066
epoch -->  4
AVG_Loss = 6.246938818613688
epoch -->  5
AVG_Loss = 6.06383725865682
epoch -->  6
AVG_Loss = 5.902800996780395
epoch -->  7
AVG_Loss = 5.756746260960897
epoch -->  8
AVG_Loss = 5.621540352503459
epoch -->  9
AVG_Loss = 5.492059373219808
epoch -->  10
AVG_Loss = 5.372639558156331
epoch -->  11
AVG_Loss = 5.254162073771159
epoch -->  12
AVG_Loss = 5.139456629435221
epoch -->  13
AVG_Loss = 5.029832277933757
epoch -->  14
AVG_Loss = 4.922100035349528
epoch -->  15
AVG_Loss = 4.818222249984741
epoch -->  16
AVG_Loss = 4.716770943959554
epoch -->  17
AVG_Loss = 4.615000806808472
epoch -->  18
AVG_Loss = 4.517725671132405
epoch -->  19
AVG_Loss = 4.41951311937968
epoch -->  20
AVG_Loss = 4.323137432098389
epoch -->  21
AVG_Loss = 4.228957390149435
epoch -->  22
AVG_Loss = 4.137051155408224
epoch -->  23
AVG_Loss = 4.043877700805664
epoch -->  24
AV

In [14]:
torch.save(model.state_dict(),r"saved_model\Tws.pth")

In [15]:
vocab = token.dict_size()
model = Tmr(64,vocab)

model.load_state_dict(torch.load(r"saved_model\Tws.pth",map_location="cpu"))

model = model

In [78]:
prediction_val = 0
for e,m in train_loader:
  prediction_val = e

In [79]:
print(prediction_val[0])

tensor([   0,   16, 1413,   40, 2807,  154,   51,   28, 1167,   51, 3020,    1,
           2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
           2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
           2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
           2,    2])


In [30]:
model.eval()
eng_dict,mar_dict = token.get_dict()

device = next(model.parameters()).device
start = "<sos>"
mar_input = [mar_dict[start]]

user = "you are a nice person"

user = user.replace("!","").replace("?","").replace(".","").replace("'","").lower()

user = token.inf_w2n(user)
print(user)
user_input = torch.tensor(user).reshape(1,-1)

with torch.no_grad():
  for i in range(50):
    initial = torch.tensor(mar_input)
    #a = input("enter sentence - ")
    a,b = model.embedding(user_input,initial)
    b = b.reshape(1,-1,64)
    #print(a.shape)
    #print(b.shape)
    out = model(a.reshape(1,50,-1),b)
    #print(out.shape)
    generated = out[0,-1,:].argmax(-1).item()
    val = torch.argmax(out)

    mar_input.append(generated)
    if generated == mar_dict["<eos>"]:
      break
  print(mar_input)





[0, 179, 224, 43, 391, 1013, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
[0, 176, 758, 3188, 382, 1]


In [31]:
new_dict = {val : key for key,val in mar_dict.items()}

for i in mar_input:
  print(new_dict[i])

<sos>
तुम्ही
सर्वात
महत्वाची
आहात
<eos>
